In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm

np.random.seed(42)

from nispace.datasets import fetch_reference
from nispace.workflows import group_comparison

In [ ]:
DATA_PATH = Path("../../data/df1.csv")

# NiSpace parcellation for cortex + aseg
PARCELLATION = "DesikanAseg"

# DK cortical regions
DK_REGIONS = [
    "bankssts","caudalanteriorcingulate","caudalmiddlefrontal","cuneus","entorhinal",
    "fusiform","inferiorparietal","inferiortemporal","isthmuscingulate","lateraloccipital",
    "lateralorbitofrontal","lingual","medialorbitofrontal","middletemporal","parahippocampal",
    "paracentral","parsopercularis","parsorbitalis","parstriangularis","pericalcarine",
    "postcentral","posteriorcingulate","precentral","precuneus","rostralanteriorcingulate",
    "rostralmiddlefrontal","superiorfrontal","superiorparietal","superiortemporal",
    "supramarginal","frontalpole","temporalpole","transversetemporal","insula",
]

# Common aseg structures typically used in group comparisons
# Keep this expandable if your table has more aseg structures you want.
ASEG_BASES = [
    "thalamus", "thalamusproper",
    "caudate",
    "putamen",
    "pallidum",
    "hippocampus",
    "amygdala",
    "accumbens", "accumbensarea",
]

DEMO_COLS = ["PATNO", "CONCOHORT", "age", "SEX", "agediag", "subgroup", "PRIMDIAG"]

CONTRASTS = {
    "De Novo PD vs HC": (1.0, 2.0),
    "Prodromal PD vs HC": (4.0, 2.0),
    "De Novo PD vs Prodromal PD": (1.0, 4.0),
}

GROUP_LABELS = {
    1.0: "De Novo PD",
    2.0: "HC",
    4.0: "Prodromal PD",
}

SELECTED_REFERENCE_MAPS = [
    "target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019",
    "target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021",
    "target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018",
    "target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018",
    "target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017",
    "target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015",
    "target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018",
    "target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012",
    "target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012",
    "target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012",
    "target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017",
    "target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012",
    "target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017",
    "target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017",
]

SELECTED_REFERENCE_CONTAINS = []

N_PERM = 10000

In [ ]:
# ----------------------------
# column finding helpers
# ----------------------------

def _norm(s: str) -> str:
    """Aggressive normalization for matching subject columns to NiSpace parcel labels."""
    s = str(s).strip().lower()

    s = s.replace("left", "lh").replace("right", "rh")
    s = s.replace("ctx-lh-", "lh_").replace("ctx-rh-", "rh_")
    s = s.replace("aparc_", "").replace("aseg_", "")

    # normalize punctuation first
    s = re.sub(r"[\s\-\./\+]+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")

    # remove metric words after punctuation cleanup
    s = re.sub(r"(_)?volume$", "", s)
    s = re.sub(r"(_)?vol$", "", s)

    # harmonize known aseg naming variants
    s = s.replace("thalamus_proper", "thalamusproper")
    s = s.replace("accumbens_area", "accumbensarea")
    s = s.replace("accumbensarea", "accumbensarea")
    s = s.replace("ventral_dc", "ventraldc")

    return s


def get_dk_volume_columns(df: pd.DataFrame) -> list[str]:
    """
    Return DK cortical volume columns.
    Expected common patterns:
      lh_bankssts_volume
      rh_insula_volume
      etc.
    """
    dk_region_pattern = "|".join(map(re.escape, DK_REGIONS))
    regex = re.compile(rf"(?i)^(lh|rh)_({dk_region_pattern})_volume$")
    return [c for c in df.columns if regex.match(c)]


def get_aseg_volume_columns(df: pd.DataFrame) -> list[str]:
    """
    Return subcortical aseg volume columns.
    This is intentionally permissive because aseg table exports vary a lot.
    Common examples:
      Left-Hippocampus
      Right-Putamen
      lh_hippocampus_volume
      rh_thalamus_volume
      Left-Thalamus-Proper
    """
    cols = []
    for c in df.columns:
        cn = _norm(c)

        has_hemi = cn.startswith("lh_") or cn.startswith("rh_")
        has_target = any(base in cn for base in ASEG_BASES)
        is_non_global = not any(
            bad in cn
            for bad in [
                "brainseg", "mask", "supra", "intracranial", "etiv", "total",
                "cortex", "white", "gray", "wm", "gm", "csf", "ventricle",
                "surfaceholes", "euler", "mean", "std", "snr"
            ]
        )

        if has_hemi and has_target and is_non_global:
            cols.append(c)

    return cols


def get_brain_volume_columns(df: pd.DataFrame) -> list[str]:
    """Return cortical DK volume columns + aseg subcortical volume columns."""
    dk_cols = get_dk_volume_columns(df)
    aseg_cols = get_aseg_volume_columns(df)

    out = dk_cols + [c for c in aseg_cols if c not in dk_cols]
    return out


# ----------------------------
# column -> parcel matching
# ----------------------------

def subject_col_to_key(col: str) -> str:
    """
    Convert subject dataframe column names to a normalized key that can be matched
    against NiSpace DesikanAseg parcel labels.
    """
    cn = _norm(col)

    # exact hemi_region form
    m = re.match(r"^(lh|rh)_([a-z0-9_]+)$", cn)
    if m:
        hemi, region = m.groups()
        region = region.replace("_", "")
        return f"{hemi}_{region}"

    return cn.replace("_", "", 1) if cn.startswith(("lh_", "rh_")) else cn


def reference_col_to_key(col: str) -> str:
    """
    Convert NiSpace DesikanAseg parcel names to the same normalized key space.
    Handles likely labels such as:
      hemi-L_lab-bankssts
      hemi-R_lab-insula
      Left-Hippocampus
      Right-Thalamus-Proper
      lh_bankssts
      rh_putamen
    """
    s = str(col)

    # NiSpace cortical style: hemi-L_lab-bankssts
    m1 = re.match(r"^hemi-([LR])_lab-(.+)$", s)
    if m1:
        hemi, region = m1.groups()
        hemi = "lh" if hemi == "L" else "rh"
        region = _norm(region).replace("_", "")
        return f"{hemi}_{region}"

    sn = _norm(s)

    m2 = re.match(r"^(lh|rh)_(.+)$", sn)
    if m2:
        hemi, region = m2.groups()
        region = region.replace("_", "")
        return f"{hemi}_{region}"

    return sn.replace("_", "")


def build_subject_to_reference_mapping(
    subject_cols: list[str],
    reference_cols: list[str],
    verbose: bool = True
) -> dict[str, str]:
    """
    Build mapping from subject dataframe columns to NiSpace reference columns
    by matching normalized keys.
    """
    subj_keys = {c: subject_col_to_key(c) for c in subject_cols}
    ref_keys = {c: reference_col_to_key(c) for c in reference_cols}

    rev_ref = {}
    for ref_col, key in ref_keys.items():
        rev_ref.setdefault(key, []).append(ref_col)

    mapping = {}
    unmatched_subject = []
    ambiguous_subject = []

    for sub_col, key in subj_keys.items():
        matches = rev_ref.get(key, [])
        if len(matches) == 1:
            mapping[sub_col] = matches[0]
        elif len(matches) == 0:
            unmatched_subject.append((sub_col, key))
        else:
            ambiguous_subject.append((sub_col, key, matches))

    if verbose:
        print(f"Matched subject columns: {len(mapping)} / {len(subject_cols)}")
        if unmatched_subject:
            print("\nUnmatched subject columns:")
            for c, k in unmatched_subject[:20]:
                print(f" - {c} -> {k}")
        if ambiguous_subject:
            print("\nAmbiguous subject columns:")
            for c, k, m in ambiguous_subject[:20]:
                print(f" - {c} -> {k} matched {m}")

    return mapping


# ----------------------------
# design / matrix preparation
# ----------------------------

def pick_etiv_column(df: pd.DataFrame) -> str:
    """
    Find eTIV column in the subject dataframe.
    Common names:
      eTIV
      EstimatedTotalIntraCranialVol
      IntraCranialVol
      intracranialvolume
    """
    candidates = [
        "eTIV",
        "EstimatedTotalIntraCranialVol",
        "IntraCranialVol",
        "intracranialvolume",
        "EstimatedTotalIntracranialVol",
    ]

    exact = [c for c in candidates if c in df.columns]
    if exact:
        return exact[0]

    # fallback fuzzy match
    for c in df.columns:
        cn = _norm(c)
        if cn in {"etiv", "estimatedtotalintracranialvol", "intracranialvol", "intracranialvolume"}:
            return c

    raise ValueError("Could not find an eTIV column in the dataframe.")


def prepare_brain_and_design(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Prepare subject-by-parcel matrix Y and design matrix.
    Uses:
      - DK cortical volumes
      - aseg subcortical volumes
      - eTIV as covariate
    """
    brain_cols = get_brain_volume_columns(df)
    etiv_col = pick_etiv_column(df)

    print("Number of cortical DK volume columns:", len(get_dk_volume_columns(df)))
    print("Number of aseg subcortical volume columns:", len(get_aseg_volume_columns(df)))
    print("Total brain volume columns:", len(brain_cols))
    print("Using eTIV column:", etiv_col)

    fs_col = "Field Strength"
    fs_cols = [fs_col] if fs_col in df.columns else []
    req_cols = ["PATNO", "CONCOHORT", "age", "SEX", etiv_col] + fs_cols

    keep_cols = req_cols + brain_cols
    df_sub = df[keep_cols].copy()

    # Y = parcel data
    Y = df_sub[brain_cols].apply(pd.to_numeric, errors="coerce")
    Y.index = df_sub["PATNO"].astype(str)

    # design = covariates + grouping
    design = pd.DataFrame(index=Y.index)
    design["CONCOHORT"] = pd.to_numeric(df_sub["CONCOHORT"], errors="coerce").values
    design["age"] = pd.to_numeric(df_sub["age"], errors="coerce").values
    design["SEX"] = df_sub["SEX"].values
    design["eTIV"] = pd.to_numeric(df_sub[etiv_col], errors="coerce").values
    if fs_col in df_sub.columns:
        fs_vals = pd.to_numeric(df_sub[fs_col], errors="coerce")
        if fs_vals.isna().all():  # string column ("1.5T" / "3T") — encode as codes
            fs_vals = pd.Categorical(df_sub[fs_col]).codes.astype(float)
            fs_vals[fs_vals < 0] = np.nan
        design["field_strength"] = fs_vals.values
    else:
        print("WARNING: 'Field Strength' column not found — covariate omitted.")

    return Y, design


def align_y_to_reference(Y: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename and reorder subject parcel columns to match NiSpace reference parcel columns.
    """
    mapping = build_subject_to_reference_mapping(
        subject_cols=Y.columns.tolist(),
        reference_cols=ref_df.columns.tolist(),
        verbose=True
    )

    Y_tmp = Y.rename(columns=mapping).copy()
    Y_aligned = Y_tmp.reindex(columns=ref_df.columns)

    print("\nY aligned shape:", Y_aligned.shape)
    print("Any missing values after parcel reindex?", Y_aligned.isna().any().any())

    missing_cols = Y_aligned.columns[Y_aligned.isna().all(axis=0)].tolist()
    if missing_cols:
        print("\nReference parcels entirely missing from subject data:")
        for col in missing_cols[:50]:
            print(" -", col)

    return Y_aligned


def build_nispace_design(d_sub: pd.DataFrame, g1: float, g2: float) -> pd.DataFrame:
    """
    Build design dataframe for NiSpace:
      groups: 0 = g1, 1 = g2
      covariates: age, SEX, eTIV
    """
    groups01 = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1}).astype(int)

    design_dict = {
        "groups": groups01,
        "age": pd.to_numeric(d_sub["age"], errors="coerce"),
        "SEX": d_sub["SEX"],
        "eTIV": pd.to_numeric(d_sub["eTIV"], errors="coerce"),
    }
    if "field_strength" in d_sub.columns:
        design_dict["field_strength"] = pd.to_numeric(d_sub["field_strength"], errors="coerce")

    design_df = pd.DataFrame(design_dict, index=d_sub.index)

    if design_df["SEX"].dtype == object:
        design_df["SEX"] = pd.Categorical(design_df["SEX"]).codes

    design_df["SEX"] = pd.to_numeric(design_df["SEX"], errors="coerce")

    return design_df

In [ ]:
def run_parcelwise_ttest(
    Y_aligned: pd.DataFrame,
    design: pd.DataFrame,
    g1: float,
    g2: float,
) -> pd.DataFrame:
    """
    Parcelwise group comparison adjusted for all covariates using OLS:
        parcel ~ group + eTIV + age + SEX

    Output matches the old function style:
        Tvalue, pvalue, df, hemi
    """

    # keep only subjects in requested contrast
    mask = design["CONCOHORT"].astype(float).isin([g1, g2])

    y_sub = Y_aligned.loc[mask].copy()
    d_sub = design.loc[mask].copy()

    # code groups as 0/1
    d_sub = d_sub.copy()
    d_sub["group01"] = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1})

    # encode SEX if needed
    if d_sub["SEX"].dtype == object:
        d_sub["SEX"] = pd.Categorical(d_sub["SEX"]).codes

    d_sub["SEX"] = pd.to_numeric(d_sub["SEX"], errors="coerce")
    d_sub["age"] = pd.to_numeric(d_sub["age"], errors="coerce")
    d_sub["eTIV"] = pd.to_numeric(d_sub["eTIV"], errors="coerce")
    d_sub["group01"] = pd.to_numeric(d_sub["group01"], errors="coerce")
    has_fs = "field_strength" in d_sub.columns
    if has_fs:
        d_sub["field_strength"] = pd.to_numeric(d_sub["field_strength"], errors="coerce")

    def parcel_hemi(parcel_name: str) -> str:
        p = str(parcel_name)
        if p.startswith("hemi-L") or p.lower().startswith("left") or p.lower().startswith("lh_"):
            return "L"
        if p.startswith("hemi-R") or p.lower().startswith("right") or p.lower().startswith("rh_"):
            return "R"
        return "NA"

    t_vals = []
    p_vals = []
    dfs = []

    for parcel in y_sub.columns:
        tmp_dict = {
            "y": pd.to_numeric(y_sub[parcel], errors="coerce"),
            "group01": d_sub["group01"],
            "eTIV": d_sub["eTIV"],
            "age": d_sub["age"],
            "SEX": d_sub["SEX"],
        }
        if has_fs:
            tmp_dict["field_strength"] = d_sub["field_strength"]
        tmp = pd.DataFrame(tmp_dict, index=y_sub.index).dropna()

        # require enough data and both groups present
        if tmp.shape[0] < 5 or tmp["group01"].nunique() < 2:
            t_vals.append(np.nan)
            p_vals.append(np.nan)
            dfs.append(np.nan)
            continue

        cov_cols = ["group01", "eTIV", "age", "SEX"] + (["field_strength"] if has_fs else [])
        X = sm.add_constant(tmp[cov_cols])
        model = sm.OLS(tmp["y"], X).fit()

        t_vals.append(model.tvalues.get("group01", np.nan))
        p_vals.append(model.pvalues.get("group01", np.nan))
        dfs.append(model.df_resid)

    df_out = pd.DataFrame(
        {
            "Tvalue": t_vals,
            "pvalue": p_vals,
            "df": dfs,
            "hemi": [parcel_hemi(c) for c in Y_aligned.columns],
        },
        index=Y_aligned.columns,
    )

    return df_out

A couple of notes:
- Tvalue and pvalue now refer to the group effect after adjusting for eTIV, age, and SEX.
- With coding {g1: 0, g2: 1}, a positive t-value means higher adjusted volume in g2 than g1.

In [ ]:
def run_group_comparisons(
    Y_aligned: pd.DataFrame,
    design: pd.DataFrame,
    ref_df: pd.DataFrame,
    contrasts: dict[str, tuple[float, float]],
    labels: dict[float, str],
    n_perm: int = 10000,
) -> tuple[pd.DataFrame, dict]:

    all_rows = []
    outputs = {}

    for contrast_name, (g1, g2) in contrasts.items():
        mask = design["CONCOHORT"].astype(float).isin([g1, g2])

        y_sub = Y_aligned.loc[mask].copy()
        d_sub = design.loc[mask].copy()

        design_df = build_nispace_design(d_sub, g1, g2)

        covariate_cols = [c for c in ["groups", "age", "SEX", "eTIV", "field_strength"]
                          if c in design_df.columns]
        valid_parcels = y_sub.columns[~y_sub.isna().all(axis=0)]
        keep = ~(
            y_sub[valid_parcels].isna().any(axis=1)
            | design_df[covariate_cols].isna().any(axis=1)
        )

        y_sub = y_sub.loc[keep]
        design_df = design_df.loc[keep]
        d_sub = d_sub.loc[keep]

        print("\n--------------------------")
        print(f"Contrast: {contrast_name}")
        print("Counts:", d_sub["CONCOHORT"].astype(float).map(labels).value_counts().to_dict())
        print("Y shape:", y_sub.shape)
        print("Design shape:", design_df.shape)
        print("Reference maps:", len(ref_df.index))

        np.random.seed(42)

        colocs, pvals, qvals, nsp = group_comparison(
            y=y_sub,
            x=ref_df,
            parcellation=PARCELLATION,
            design=design_df,
            comparison_method="hedges(a,b)",
            colocalization_method="spearman",
            n_perm=n_perm,
            n_proc=-1,
            verbose=True,
        )

        outputs[contrast_name] = {
            "colocs": colocs,
            "p": pvals,
            "q": qvals,
            "nsp": nsp,
        }

        rho = np.asarray(colocs).ravel()
        p = np.asarray(pvals).ravel()
        q = np.asarray(qvals).ravel()

        df_out = pd.DataFrame(
            {
                "reference_map": ref_df.index,
                "rho": rho,
                "p": p,
                "q": q,
                "contrast": contrast_name,
            }
        )

        all_rows.append(df_out)

    df_all = pd.concat(all_rows, ignore_index=True).sort_values(["contrast", "q", "p"])
    return df_all, outputs

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

Y, design = prepare_brain_and_design(df)

print("\nY shape:", Y.shape)
print("Design shape:", design.shape)
print("\nCONCOHORT counts:")
print(design["CONCOHORT"].value_counts(dropna=False))

In [ ]:
### Fetch and select DesikanAseg reference maps
### right parcellation call for combined cortex + aseg

df_reference_desikan_aseg = fetch_reference(
    "pet",
    collection="UniqueTracers",
    parcellation=PARCELLATION,
    print_references=True,
)

print("\nFull reference shape:", df_reference_desikan_aseg.shape)
print("Available reference maps:")
print(df_reference_desikan_aseg.index.tolist())

df_reference_selected = df_reference_desikan_aseg[
    df_reference_desikan_aseg.index.get_level_values("map").isin(SELECTED_REFERENCE_MAPS)
]

if SELECTED_REFERENCE_CONTAINS:
    mask = df_reference_selected.index.get_level_values("map").str.contains(
        "|".join(SELECTED_REFERENCE_CONTAINS),
        case=False,
        na=False
    )
    df_reference_selected = df_reference_selected[mask]

reference_table = df_reference_selected.index.to_frame(index=False)
reference_table.columns = ["Set", "Map"]
reference_table = reference_table.reset_index(drop=True)
reference_table.index = reference_table.index + 1

print("\nSelected reference shape:", df_reference_selected.shape)
print("Selected reference maps:")
print(reference_table.to_string())

print("\nReference parcel order preview:")
print(df_reference_selected.columns.tolist()[:20])

In [ ]:
Y_aligned = align_y_to_reference(Y, df_reference_selected)

common_idx = Y_aligned.index.intersection(design.index)
Y_aligned = Y_aligned.loc[common_idx].copy()
design = design.loc[common_idx].copy()

print("\nFinal aligned Y shape:", Y_aligned.shape)
print("Final aligned design shape:", design.shape)


In [ ]:
df_ggseg = (
    design.loc[common_idx, [c for c in ["CONCOHORT", "age", "SEX", "eTIV", "field_strength"]
                             if c in design.columns]]
    .copy()
    .assign(PATNO=common_idx)
)

_demo_cols = [c for c in ["PATNO", "CONCOHORT", "age", "SEX", "eTIV", "field_strength"]
              if c in df_ggseg.columns]
df_ggseg = pd.concat(
    [
        df_ggseg[_demo_cols],
        Y_aligned
    ],
    axis=1
)

df_ggseg.to_csv("../../data/df1_id_volume_desikanaseg_aligned.csv", index=False)
print("Saved: df1_id_volume_desikanaseg_aligned.csv")
print(df_ggseg.shape)
display(df_ggseg.head())

In [ ]:
parcelwise_results = {}

for contrast_name, (g1, g2) in CONTRASTS.items():
    df_res = run_parcelwise_ttest(Y_aligned, design, g1, g2)

    # reset index and keep clean structure
    df_res = df_res.reset_index().rename(columns={"index": "parcel"})

    # optional: keep only useful columns (if you added beta_group, include it)
    keep_cols = [c for c in ["parcel", "beta_group", "Tvalue", "pvalue", "df", "hemi"] if c in df_res.columns]
    df_res = df_res[keep_cols]

    # sort once and store sorted version
    df_res_sorted = df_res.sort_values("pvalue").reset_index(drop=True)

    # store
    parcelwise_results[contrast_name] = df_res_sorted

    # print nicely
    print("\n" + "="*40)
    print(f"Top 5 parcels for: {contrast_name}")
    print("="*40)
    display(df_res_sorted.head(5))

In [ ]:
df_all, outputs = run_group_comparisons(
    Y_aligned=Y_aligned,
    design=design,
    ref_df=df_reference_selected,
    contrasts=CONTRASTS,
    labels=GROUP_LABELS,
    n_perm=N_PERM,
)

In [ ]:
print("\n=============================")
print("Combined results")
print("=============================")
display(df_all.head(10))

df_all.to_csv("../../results/nispace_group_comparison_results_volumes_desikanaseg.csv", index=False)
print("Saved: nispace_group_comparison_results_volumes_desikanaseg.csv")

for contrast_name, res in outputs.items():
    print("\n=============================")
    print(f"Contrast: {contrast_name}")
    print("=============================")

    colocs = res["colocs"]
    pvals = res["p"]
    qvals = res["q"]

    print("Top 5 lowest q-values")
    print(qvals.T["mean"].sort_values().head(5))

    display("Colocalization:", colocs)
    display("p values:", pvals)
    display("q values:", qvals)

print("\nTotal NaNs in selected reference:", df_reference_selected.isna().sum().sum())
print("NaNs per selected map:")
display(df_reference_selected.isna().sum(axis=1).sort_values(ascending=False))

for contrast_name in df_all["contrast"].unique():
    print(f"\nTop 5 results for {contrast_name}:")
    display(
        df_all[df_all["contrast"] == contrast_name]
        .sort_values("q")
        .head(5)
    )

In [ ]:
zscore_outputs = {}

for contrast_name, (gA, gB) in CONTRASTS.items():
    mask = design["CONCOHORT"].astype(float).isin([gA, gB])
    y = Y_aligned.loc[mask].copy()
    d = design.loc[mask].copy()

    design_sub = build_nispace_design(d, gA, gB)

    # Exclude parcels that are all-NaN (removed structures like VentralDC) from row filter
    valid_parcels = y.columns[~y.isna().all(axis=0)]
    keep = ~(
        y[valid_parcels].isna().any(axis=1)
        | design_sub[[c for c in ["groups", "age", "SEX", "eTIV", "field_strength"] if c in design_sub.columns]].isna().any(axis=1)
    )

    y = y.loc[keep]
    d = d.loc[keep]
    design_sub = design_sub.loc[keep]

    print(f"\n{contrast_name}")
    print("Counts:", d["CONCOHORT"].astype(float).map(GROUP_LABELS).value_counts().to_dict())

    colocs, pvals, qvals, nsp = group_comparison(
        y=y,
        x=df_reference_selected,
        parcellation=PARCELLATION,
        design=design_sub,
        comparison_method="zscore(a,b)",
        colocalization_method="spearman",
        n_perm=N_PERM,
        n_proc=-1,
        verbose=False,
        plot_design=False,
    )

    zscore_outputs[contrast_name] = {
        "colocs": colocs,
        "p": pvals,
        "q": qvals,
        "nsp": nsp,
    }

for contrast, res in zscore_outputs.items():
    print(f"\n{contrast}")
    display("Colocalization:", res["colocs"].head(3))
    display("P values:", res["p"].head())
    display("Q values:", res["q"].head())

In [ ]:
# Build and save merged dataset for clinical correlation (volume - DesikanAseg)
all_rows = []

for contrast, res in zscore_outputs.items():
    colocs = res["colocs"].copy()

    if colocs.index.name is None:
        colocs.index.name = "PATNO"

    colocs_long = colocs.stack(list(range(colocs.columns.nlevels))).reset_index()
    colocs_long = colocs_long.rename(columns={0: "colocalization"})

    meta_cols = [c for c in colocs_long.columns if c not in ["PATNO", "colocalization"]]
    colocs_long["map"] = colocs_long[meta_cols].astype(str).agg(" | ".join, axis=1)

    colocs_long = colocs_long[["PATNO", "map", "colocalization"]].copy()
    colocs_long["contrast"] = contrast

    all_rows.append(colocs_long)

nispace_df = pd.concat(all_rows, ignore_index=True)

print(nispace_df.head())
print(nispace_df.shape)
print(nispace_df["contrast"].unique())

clinical_df = df[["PATNO", "updrs3_score", "moca", "gds", "SEX", "age"]].copy()

nispace_df["PATNO"] = nispace_df["PATNO"].astype(str)
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)

merged_df = nispace_df.merge(clinical_df, on="PATNO", how="inner")
merged_df.to_csv("../../data/merged_df_volume_desikanaseg.csv", index=False)

print("\nSaved as merged_df_volume_desikanaseg.csv")